# ДЗ: LLM-приложение — суммаризатор документов с мониторингом через Langfuse

**Что внутри:**
1. Установка зависимостей
2. Настройка ключей (OpenAI + Langfuse)
3. Подключение Langfuse (drop-in интеграция для OpenAI + декоратор `@observe`)
4. Загрузка документа (`.txt` / `.md` / `.pdf`)
5. Чанкинг длинного текста
6. Суммаризация по стратегии **map-reduce**
7. Просмотр трейсов в Langfuse и оценка качества (score)

Модель: **OpenAI GPT** (`gpt-4o-mini` по умолчанию). Весь пайплайн логируется в Langfuse:
входные данные, промпты, ответы, токены, стоимость и латентность.

### Сущности Langfuse и где они в этом приложении

| Сущность | Что это | Где в ноутбуке |
|---|---|---|
| **Trace** | Полный путь выполнения запроса | один вызов `summarize_document()` = один трейс |
| **Span** | Отдельная операция | функции под `@observe()`: `summarize_chunk`, `reduce_summaries` |
| **Generation** | Специальный span для вызова LLM | автоматически от `langfuse.openai` на каждый `chat.completions.create` |
| **Event** | Точечное событие | `create_event`: загрузка документа, завершение map-этапа, ошибки |
| **Score** | Метрика качества/производительности | `create_score`: ручная оценка + авто-метрика степени сжатия |

## 1. Установка зависимостей

In [ ]:
# Запусти один раз. Если пакеты уже стоят — ячейку можно пропустить.
%pip install -q -r requirements.txt

## 2. Настройка ключей

Ключи берутся из файла `.env` (скопируй `.env.example` -> `.env` и заполни).
Если `.env` нет — ячейка запросит ключи интерактивно через `getpass`.

In [ ]:
import os
from getpass import getpass

try:
    from dotenv import load_dotenv
    load_dotenv()  # подхватит .env, если он есть
except ImportError:
    pass

def ensure(var, prompt, secret=True):
    """Гарантирует, что переменная окружения задана."""
    if not os.environ.get(var):
        os.environ[var] = getpass(prompt) if secret else input(prompt)
    return os.environ[var]

ensure('OPENAI_API_KEY',    'OpenAI API key: ')
ensure('LANGFUSE_PUBLIC_KEY','Langfuse PUBLIC key (pk-lf-...): ')
ensure('LANGFUSE_SECRET_KEY','Langfuse SECRET key (sk-lf-...): ')
# Регион Langfuse: EU -> https://cloud.langfuse.com, US -> https://us.cloud.langfuse.com
os.environ.setdefault('LANGFUSE_HOST', 'https://cloud.langfuse.com')

print('Ключи заданы. Langfuse host:', os.environ['LANGFUSE_HOST'])

## 3. Подключение Langfuse

Используем два механизма Langfuse:
- **drop-in интеграцию** `from langfuse.openai import openai` — это обёртка над
  официальным клиентом OpenAI, которая автоматически логирует каждый вызов модели
  (промпт, ответ, токены, стоимость, латентность) как *generation* в трейсе;
- **декоратор `@observe()`** — оборачивает наши функции в *spans*, формируя
  иерархию трейса (видно отдельные шаги map и финальный reduce).

In [ ]:
# ВАЖНО: импортируем openai именно из langfuse, чтобы вызовы трекались автоматически
from langfuse.openai import openai
from langfuse import observe, get_client, propagate_attributes

langfuse = get_client()

# Проверяем, что креды валидны и есть связь с сервером Langfuse
assert langfuse.auth_check(), 'Langfuse auth failed — проверь ключи и LANGFUSE_HOST'
print('Langfuse подключён успешно.')

## 4. Загрузка документа

Поддерживаются `.txt`, `.md` и `.pdf`. По умолчанию берём `sample_document.txt`
из репозитория — замени путь на свой файл при необходимости.

In [ ]:
from pathlib import Path

def load_document(path: str) -> str:
    """Читает документ и возвращает его текст."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Файл не найден: {path}')
    if path.suffix.lower() == '.pdf':
        from pypdf import PdfReader
        reader = PdfReader(str(path))
        return '\n'.join((page.extract_text() or '') for page in reader.pages)
    # txt / md и прочий текст
    return path.read_text(encoding='utf-8')

DOC_PATH = 'sample_document.txt'  # <-- замени на свой документ
document = load_document(DOC_PATH)

print(f'Документ: {DOC_PATH}')
print(f'Символов: {len(document)}')
print('--- начало ---')
print(document[:500], '...')

## 5. Чанкинг

Если документ длиннее контекстного окна, его нужно разбить на части. Считаем размер
в токенах через `tiktoken` и режем с небольшим перекрытием, чтобы не рвать смысл.

In [ ]:
import tiktoken

ENC = tiktoken.get_encoding('cl100k_base')

def count_tokens(text: str) -> int:
    return len(ENC.encode(text))

def chunk_text(text: str, max_tokens: int = 800, overlap: int = 100):
    """Делит текст на чанки по числу токенов с перекрытием."""
    tokens = ENC.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk = ENC.decode(tokens[start:end])
        chunks.append(chunk)
        start += max_tokens - overlap
    return chunks

chunks = chunk_text(document)
print(f'Всего токенов: {count_tokens(document)}')
print(f'Чанков: {len(chunks)}')

## 6. Суммаризация (map-reduce) с мониторингом

- `summarize_chunk` — суммаризирует один фрагмент (этап **map**);
- `reduce_summaries` — объединяет промежуточные резюме в финальное (этап **reduce**);
- `summarize_document` — оркестратор; помечен `@observe()`, поэтому в Langfuse
  создаётся единый трейс со вложенными шагами.

Каждый вызов `openai.chat.completions.create` логируется автоматически.

In [ ]:
MODEL = 'gpt-4o-mini'

def _chat(system: str, user: str, temperature: float = 0.2) -> str:
    resp = openai.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': user},
        ],
    )
    return resp.choices[0].message.content.strip()

@observe()
def summarize_chunk(chunk: str) -> str:
    """MAP: краткое резюме одного фрагмента."""
    return _chat(
        'Ты — ассистент-суммаризатор. Кратко и точно изложи суть фрагмента на русском.',
        f'Суммаризируй фрагмент документа:\n\n{chunk}',
    )

@observe()
def reduce_summaries(summaries: list[str]) -> str:
    """REDUCE: объединяет резюме фрагментов в финальное структурированное резюме."""
    joined = '\n\n'.join(f'- {s}' for s in summaries)
    return _chat(
        'Ты — ассистент-суммаризатор. Сформируй цельное структурированное резюме на русском: '
        '2-3 предложения общего вывода, затем маркированный список ключевых тезисов.',
        f'Объедини промежуточные резюме в одно финальное:\n\n{joined}',
    )

# id последнего трейса — пригодится, чтобы прикрепить ссылку и оценку
LAST_TRACE_ID = None

@observe(name='document-summarization')
def summarize_document(text: str) -> str:
    """Полный пайплайн map-reduce. Один трейс на весь документ.
    @observe автоматически логирует вход (text) и выход (резюме)."""
    global LAST_TRACE_ID
    parts = chunk_text(text)
    # propagate_attributes — рекомендуемый в Langfuse v3+ способ задать
    # теги/метаданные трейса; они проставятся и на все вложенные шаги map/reduce.
    with propagate_attributes(
        tags=['hw12', 'summarizer'],
        metadata={'doc': DOC_PATH, 'chunks': len(parts), 'tokens': count_tokens(text)},
    ):
        # EVENT: точечное событие загрузки документа
        langfuse.create_event(
            name='document-loaded',
            input={'doc': DOC_PATH},
            metadata={'chars': len(text), 'tokens': count_tokens(text), 'chunks': len(parts)},
        )
        if len(parts) == 1:
            result = summarize_chunk(parts[0])
        else:
            partial = [summarize_chunk(p) for p in parts]
            # EVENT: завершение map-этапа
            langfuse.create_event(name='map-stage-complete', metadata={'chunks': len(parts)})
            result = reduce_summaries(partial)
    LAST_TRACE_ID = langfuse.get_current_trace_id()
    return result

In [ ]:
summary = summarize_document(document)
print(summary)

## 7. Трейсы в Langfuse и оценка качества

`langfuse.flush()` досылает буфер событий на сервер. Затем получаем ссылку на трейс
и можем прикрепить к нему оценку (`score`) — например, ручную оценку качества резюме.

In [ ]:
# Досылаем буфер событий на сервер Langfuse
langfuse.flush()

trace_id = LAST_TRACE_ID
print('trace_id:', trace_id)
try:
    print('URL:', langfuse.get_trace_url(trace_id=trace_id))
except Exception as e:
    print('Открой трейс в дашборде Langfuse вручную. (', e, ')')

In [ ]:
# SCORE 1 — ручная оценка качества (0..1). На практике сюда кладут оценку
# человека или результат авто-оценщика (LLM-as-a-judge).
langfuse.create_score(
    trace_id=trace_id,
    name='manual-quality',
    value=0.9,
    comment='Резюме точное и связное',
)

# SCORE 2 — авто-метрика производительности: степень сжатия документа
compression = round(count_tokens(summary) / count_tokens(document), 3)
langfuse.create_score(
    trace_id=trace_id,
    name='compression-ratio',
    value=compression,
    comment=f'итог {count_tokens(summary)} токенов из {count_tokens(document)}',
)

langfuse.flush()
print(f'Scores отправлены (compression-ratio={compression}). Открой трейс в Langfuse.')

### Event уровня ERROR

`Event` фиксирует и ошибки. Ниже — намеренно неверный вызов модели; исключение
перехватывается и логируется как событие с `level="ERROR"`, привязанное к трейсу.

In [ ]:
@observe(name='summarization-with-error-handling')
def safe_summarize(text: str):
    try:
        return _chat('Ты — суммаризатор.', text, temperature=0.2)
    except Exception as e:
        # EVENT с уровнем ERROR — будет видно в трейсе как ошибка
        langfuse.create_event(
            name='llm-call-failed',
            level='ERROR',
            status_message=str(e),
            metadata={'model': MODEL},
        )
        raise

# Демонстрация: ломаем имя модели, чтобы поймать и залогировать ошибку
_real_model = MODEL
try:
    MODEL = 'gpt-does-not-exist'
    try:
        safe_summarize('Короткий тестовый текст для проверки обработки ошибки.')
    except Exception as e:
        print('Ошибка перехвачена и залогирована как Event(ERROR):', type(e).__name__)
finally:
    MODEL = _real_model
langfuse.flush()

## 8. Datasets — тестовый набор для оценки

**Dataset** в Langfuse — это набор пар «вход → ожидаемый результат», по которому
можно воспроизводимо прогонять приложение и сравнивать версии (регрессионное
тестирование качества). Создадим небольшой датасет из коротких текстов; для каждого
зададим ключевые темы, которые обязаны попасть в резюме.

In [ ]:
DATASET_NAME = 'hw12-summarization-eval'

# Создаём датасет (идемпотентно: повторный вызов не создаёт дубликат)
langfuse.create_dataset(
    name=DATASET_NAME,
    description='Короткие тексты для оценки качества суммаризатора',
)

# Элементы: input — документ, expected_output — ключевые темы для проверки покрытия
items = [
    {'doc': 'Фотосинтез — процесс, при котором растения преобразуют солнечный свет, '
            'воду и углекислый газ в глюкозу и кислород. Он лежит в основе пищевых '
            'цепей и насыщает атмосферу кислородом.',
     'keywords': ['солнечн', 'кислород', 'углекислый', 'глюкоз']},
    {'doc': 'Чёрная дыра — область пространства-времени с гравитацией настолько '
            'сильной, что её не покидает даже свет. Образуется при коллапсе массивной '
            'звезды. Её граница называется горизонтом событий.',
     'keywords': ['гравитац', 'свет', 'горизонт событий', 'звезд']},
    {'doc': 'HTTP — протокол прикладного уровня для передачи гипертекста в вебе. '
            'Работает по модели запрос-ответ между клиентом и сервером. HTTPS '
            'добавляет шифрование через TLS.',
     'keywords': ['протокол', 'запрос', 'сервер', 'шифрован']},
]

for i, it in enumerate(items):
    langfuse.create_dataset_item(
        dataset_name=DATASET_NAME,
        id=f'{DATASET_NAME}-{i}',  # фиксированный id => повторный запуск перезапишет, без дублей
        input={'document': it['doc']},
        expected_output={'keywords': it['keywords']},
    )
print(f"Датасет '{DATASET_NAME}' наполнен: {len(items)} элементов")

## 9. Эксперимент с evaluator

**Эксперимент** прогоняет `task` по всем элементам датасета и применяет к каждому
результату один или несколько **evaluator**'ов. Здесь — два кастомных evaluator'а:

- `keyword-coverage` — доля ожидаемых ключевых тем, попавших в резюме (качество);
- `conciseness` — краткость резюме по числу токенов (производительность).

> Можно подключить и готовый evaluator: `from langfuse.experiment import
> create_evaluator_from_autoevals` (нужен пакет `autoevals`) — например, оценщики
> `Factuality` или `Summary` на базе LLM-as-a-judge.

In [ ]:
from langfuse import Evaluation

# TASK: что делаем с каждым элементом датасета
def summarization_task(*, item, **kwargs):
    document = item.input['document']
    return _chat('Ты — суммаризатор. Сожми текст до 1-2 предложений на русском.', document)

# EVALUATOR 1 (кастомный): покрытие ожидаемых ключевых тем
def keyword_coverage(*, input, output, expected_output, metadata=None, **kwargs):
    keywords = (expected_output or {}).get('keywords', [])
    if not keywords:
        return Evaluation(name='keyword-coverage', value=0.0, comment='нет эталона')
    low = output.lower()
    hits = [k for k in keywords if k.lower() in low]
    return Evaluation(
        name='keyword-coverage',
        value=round(len(hits) / len(keywords), 3),
        comment=f'найдено {len(hits)}/{len(keywords)}: {hits}',
    )

# EVALUATOR 2 (кастомный): краткость (1.0 если <= 60 токенов)
def conciseness(*, input, output, expected_output=None, metadata=None, **kwargs):
    n = count_tokens(output)
    value = 1.0 if n <= 60 else round(60 / n, 3)
    return Evaluation(name='conciseness', value=value, comment=f'{n} токенов')

In [ ]:
dataset = langfuse.get_dataset(DATASET_NAME)

result = dataset.run_experiment(
    name='summarizer-v1',
    description='map-reduce суммаризатор на gpt-4o-mini',
    task=summarization_task,
    evaluators=[keyword_coverage, conciseness],
)

langfuse.flush()
# Сводка по эксперименту (средние значения метрик и пр.)
print(result.format())

## Итоги

- Построен **суммаризатор документов** на OpenAI GPT с пайплайном **map-reduce**.
- Продемонстрированы все ключевые сущности Langfuse:
  - **Trace** — единый трейс на вызов `summarize_document`;
  - **Span** — шаги `summarize_chunk` / `reduce_summaries` под `@observe`;
  - **Generation** — авто-логирование вызовов LLM обёрткой `langfuse.openai`;
  - **Event** — `document-loaded`, `map-stage-complete`, `llm-call-failed` (ERROR);
  - **Score** — `manual-quality` и авто-метрика `compression-ratio`.
- Освоены дополнительные механизмы:
  - **Dataset** `hw12-summarization-eval` — тестовый набор «вход → ожидаемые темы»;
  - **Experiment** `summarizer-v1` с кастомными evaluator'ами `keyword-coverage` и
    `conciseness` (плюс заметка о готовых evaluator'ах через `autoevals`).

В дашборде Langfuse доступны: список трейсов с тегом `hw12`, разбивка по токенам и
стоимости, латентность каждого шага, точечные события, оценки, а также датасет и
страница эксперимента со сравнением метрик по прогонам.